# Loading Data

In [18]:
import pandas as pd
import ast
import numpy as np

In [19]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/dnb-handle-data")

print("Path to dataset files:", path)

100%|██████████| 312k/312k [00:00<00:00, 36.4MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/s1nghayush/dnb-handle-data/versions/3


In [20]:
mbm_bowl = pd.read_csv(path + '/ganjoo_ipl_mbm_bowl.csv')
playing_xi = pd.read_csv(path + '/playing_xi.csv')
player_info = pd.read_csv(path + '/ganjoo_player_id_mapping.csv')

# Cleaning Data

In [21]:
# Update team names in team1_name and team2_name columns
playing_xi['team1_name'] = playing_xi['team1_name'].replace({'Delhi Daredevils': 'Delhi Capitals',
                                                               'Kings XI Punjab': 'Punjab Kings',
                                                            'Rising Pune Supergiants': 'Rising Pune Supergiant'})
playing_xi['team2_name'] = playing_xi['team2_name'].replace({'Delhi Daredevils': 'Delhi Capitals',
                                                               'Kings XI Punjab': 'Punjab Kings',
                                                            'Rising Pune Supergiants': 'Rising Pune Supergiant'})

In [22]:
mbm_bowl['bowler_id'] = mbm_bowl['bowler_id'].astype(str)

def parse_playing_xi(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

# Apply the conversion on both team1 and team2 playing XI columns
playing_xi['team1_playing_xi'] = playing_xi['team1_playing_xi'].apply(parse_playing_xi)
playing_xi['team2_playing_xi'] = playing_xi['team2_playing_xi'].apply(parse_playing_xi)

# Create player mapping dictionary (key: player_id as string, value: player_name)
player_map = dict(zip(player_info['player_id'].astype(str), player_info['player_name']))

# handling dnb values

In [23]:
dnb_bowl_rows = []

# Group by match_id and innings
for (match_id, inns), group in mbm_bowl.groupby(['match_id', 'inns']):
    # Extract match details
    date = group['date'].iloc[0]
    bat_team = group['bat_team'].iloc[0]   # Batting team
    bowl_team = group['bowl_team'].iloc[0] # Bowling team
    venue = group['venue'].iloc[0]

    # Get set of bowler IDs who actually bowled
    bowled_ids = set(group['bowler_id'].unique())

    # Lookup the playing XI row for the match_id
    xi_row = playing_xi[playing_xi['match_id'] == match_id]
    if xi_row.empty:
        print(f"Warning: Match ID {match_id} not found in playing XI data.")
        continue

    row = xi_row.iloc[0]

    # Get the playing XI for the bowling team
    if row['team1_name'] == bowl_team:
        full_xi = row['team1_playing_xi']
    elif row['team2_name'] == bowl_team:
        full_xi = row['team2_playing_xi']
    else:
        print(f"Warning: Bowling team {bowl_team} not found in playing XI teams for match {match_id}.")
        continue

    # Convert playing XI player IDs to strings
    full_xi = [str(pid) for pid in full_xi]

    # Identify players in playing XI who did not bowl
    dnb_bowlers = [pid for pid in full_xi if pid not in bowled_ids]

    # Add a new row for each bowler who did not bowl
    for pid in dnb_bowlers:
        bowler_name = player_map.get(pid, "")  # Get player name if available
        new_row = {
            'date': date,
            'match_id': match_id,
            'bowl_team': bowl_team,
            'bat_team': bat_team,
            'bowler_id': pid,
            'bowler': bowler_name,
            'inns': inns,
            'overs': 'DNB',
            'runs': 0,
            'wickets': 0,
            'maiden': 0,
            'eco': 0,
            'balls':0,
            'lbw_or_bowled': 0,
            'dots':0,
            'venue': venue
        }
        dnb_bowl_rows.append(new_row)

In [24]:
new_df = pd.DataFrame(dnb_bowl_rows)

In [25]:
combined_df = pd.concat([mbm_bowl, new_df], ignore_index=True)

In [26]:
combined_df['bowler'] = combined_df.apply(
    lambda row: row['bowler'] if pd.notnull(row['bowler']) and row['bowler'] != ""
    else player_map.get(row['bowler_id'], ""), axis=1
)

combined_df['is_DNB_Bowl'] = combined_df['overs'].apply(lambda x: 1 if x == "DNB" else 0)
combined_df = combined_df.sort_values(by=['match_id', 'inns', 'is_DNB_Bowl']).reset_index(drop=True)
combined_df.drop(columns=['is_DNB_Bowl'], inplace=True)


In [28]:
combined_df[combined_df['match_id'] == 829705].sort_values(by='inns')

,date,match_id,bowl_team,bat_team,bowler_id,bowler,inns,overs,runs,wickets,maiden,eco,balls,lbw_or_bowled,dots,venue
0,08-04-2015,829705,Kolkata Knight Riders,Mumbai Indians,376116,Umesh Yadav,1,3.0,36,0,0,12.000000,18,0,6,Eden Gardens
1,08-04-2015,829705,Kolkata Knight Riders,Mumbai Indians,46538,Morne Morkel,1,4.0,18,2,1,4.500000,24,0,15,Eden Gardens
2,08-04-2015,829705,Kolkata Knight Riders,Mumbai Indians,56143,Shakib Al Hasan,1,4.0,48,1,0,12.000000,26,0,7,Eden Gardens
3,08-04-2015,829705,Kolkata Knight Riders,Mumbai Indians,230558,Sunil Narine,1,4.0,28,0,0,7.000000,24,0,10,Eden Gardens
4,08-04-2015,829705,Kolkata Knight Riders,Mumbai Indians,276298,Andre Russell,1,3.0,21,0,0,7.000000,18,0,7,Eden Gardens
5,08-04-2015,829705,Kolkata Knight Riders,Mumbai Indians,32966,Piyush Chawla,1,2.0,16,0,0,8.000000,12,0,3,Eden Gardens
6,08-04-2015,829705,Kolkata Knight Riders,Mumbai Indians,28763,Gautam Gambhir,1,DNB,0,0,0,0.000000,0,0,0,Eden Gardens
7,08-04-2015,829705,Kolkata Knight Riders,Mumbai Indians,290630,Manish Pandey,1,DNB,0,0,0,0.000000,0,0,0,Eden Gardens
8,08-04-2015,829705,Kolkata Knight Riders,Mumbai Indians,35582,Robin Uthappa,1,DNB,0,0,0,0.000000,0,0,0,Eden Gardens
9,08-04-2015,829705,Kolkata Knight Riders,Mumbai Indians,446507,Suryakumar Yadav,1,DNB,0,0,0,0.000000,0,0,0,Eden Gardens


In [29]:
combined_df.to_csv('ganjoo_ipl_mbm_bowl_final.csv', index=False)